In [1]:
#https://grouplens.org/datasets/movielens/
import pandas as pd

input_folder='ml-latest-small'

In [2]:
!unzip ml-latest-small.zip

Archive:  ml-latest-small.zip
   creating: ml-latest-small/
  inflating: ml-latest-small/links.csv  
  inflating: ml-latest-small/movies.csv  
  inflating: ml-latest-small/ratings.csv  
  inflating: ml-latest-small/README.txt  
  inflating: ml-latest-small/tags.csv  


In [3]:
movies_df=pd.read_csv(input_folder+'/movies.csv', index_col='movieId')

movies_df

,title,genres
movieId,,
1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,Jumanji (1995),Adventure|Children|Fantasy
3,Grumpier Old Men (1995),Comedy|Romance
4,Waiting to Exhale (1995),Comedy|Drama|Romance
5,Father of the Bride Part II (1995),Comedy
...,...,...
193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy
193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy
193585,Flint (2017),Drama


In [4]:
ratings_df=pd.read_csv(input_folder+'/ratings.csv')

ratings_df

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


In [5]:
def discretize_rating(rating:float):

    '''
    Converts a given float rating to a string value

    '''
    polarity='A' # average

    if rating<3: polarity='N' # negative
    elif rating>3:polarity='P' # positive

    return polarity

<h3> User-based recommendations </h3>

In [6]:
def load_user_ratings(ratings_df:pd.core.frame.DataFrame):

    '''
    Loads all the ratings submitted by each user

    Returns a dictionary that maps each user to a second dict that maps movies to discretized ratings

    '''

    distinct_users=set(ratings_df['userId']) # get all distinct users

    user_ratings={} # store movie ratings per user

    for user in distinct_users: # for each user

        # get the movie id and rating for every rating submitted by this user
        my_ratings=ratings_df[ratings_df.userId==user][['movieId','rating']]

        #discretize the ratings and attach them to the user
        user_ratings[user]=dict(zip(my_ratings.movieId, my_ratings.rating.apply(discretize_rating)))

    return user_ratings



In [7]:
user_ratings=load_user_ratings(ratings_df)

In [8]:
user_ratings[10]

{296: 'N',
 356: 'P',
 588: 'P',
 597: 'P',
 912: 'P',
 1028: 'N',
 1088: 'A',
 1247: 'A',
 1307: 'A',
 1784: 'P',
 1907: 'P',
 2571: 'N',
 2671: 'P',
 2762: 'N',
 2858: 'N',
 2959: 'N',
 3578: 'P',
 3882: 'A',
 4246: 'P',
 4306: 'P',
 4447: 'P',
 4993: 'P',
 4995: 'P',
 5066: 'A',
 5377: 'P',
 5620: 'A',
 5943: 'A',
 5952: 'P',
 5957: 'A',
 6155: 'A',
 6266: 'A',
 6377: 'P',
 6535: 'P',
 6942: 'P',
 7149: 'P',
 7151: 'A',
 7153: 'P',
 7154: 'P',
 7169: 'P',
 7293: 'P',
 7375: 'P',
 7451: 'P',
 7458: 'P',
 8529: 'P',
 8533: 'P',
 8636: 'P',
 8665: 'P',
 8808: 'A',
 8869: 'P',
 8961: 'N',
 8969: 'P',
 8970: 'N',
 30749: 'P',
 31433: 'N',
 31685: 'P',
 33145: 'A',
 33679: 'A',
 33794: 'P',
 40629: 'A',
 40819: 'P',
 41285: 'N',
 47099: 'A',
 49272: 'P',
 49286: 'P',
 51662: 'A',
 51705: 'P',
 51834: 'N',
 54286: 'P',
 56367: 'P',
 56949: 'A',
 58047: 'P',
 58559: 'P',
 59333: 'N',
 59421: 'N',
 60397: 'A',
 60950: 'N',
 61250: 'N',
 63113: 'P',
 63992: 'P',
 64969: 'N',
 66203: 'P',
 689

In [9]:
from itertools import combinations
from collections import defaultdict

def get_user_neighbors(user_ratings:dict, # ratings submitted by each user
                       min_rating_num:int=5 # at least this many ratings are required for a comparison
                      ):

    '''
    Compute rating-based similarity between every two pairs of users

    '''

    #get all possible pairs of usres
    pairs=list(combinations(list(user_ratings.keys()),2))

    usim=defaultdict(dict) # initialize the sim dictionary

    for u1,u2 in pairs: # for every user pair

        #get a set with all the discretized ratings (movie id, polarity tuples) for u1 and u2
        s1=set([(mid,pol) for mid,pol in user_ratings[u1].items()])
        s2=set([(mid,pol) for mid,pol in user_ratings[u2].items()])

        # check if both users respect the lower bound
        if len(s1)<min_rating_num or len(s2)<min_rating_num: continue

        # get the union and intersection for these two users
        union=s1.union(s2)
        inter=s1.intersection(s2)

        # compute user sim via the jaccard coeff
        jacc=len(inter)/len(union)

        # remember the sim values
        usim[u1][u2]=jacc
        usim[u2][u1]=jacc

    # attach each user to its neighbors, sorted by sim in descending order
    return {user:sorted(usim[user].items(),key=lambda x:x[1], reverse=True) for user in usim}




In [10]:
neighbors_u=get_user_neighbors(user_ratings)

In [11]:
neighbors_u[33][:10]

[(263, 0.0990990990990991),
 (144, 0.08812260536398467),
 (391, 0.08617234468937876),
 (58, 0.08502024291497975),
 (566, 0.08490566037735849),
 (304, 0.08139534883720931),
 (323, 0.08085106382978724),
 (191, 0.08071748878923767),
 (290, 0.07908163265306123),
 (336, 0.07614213197969544)]

In [12]:

def recommend_ub(user:int,
                 movies_df:pd.core.frame.DataFrame, # movie info
                 neighbors_u:dict, # neighbors dict
                 user_ratings:dict, # ratings submitted per user
                 neighbor_num:int, # number of neighbors to consider
                 rec_num:int# number of movies to recommend
                ):

    '''
    Delivers user-based recommendations. Given a specific user:
    - find the user's neighbor_num most similar users
    - Go over all the movies rated by all neighbors
    - Each movie gets +2 if a neighbor liked it, -2 if a neighbor didn't like it, -1 if  neighbor was neutral
    - +2,-1,and -2 are scaled based on user sim
    - Sort the movies by their scores in desc order
    - Go over the sorted movie list. If the user has already rated the movie, store its rating. Otherwise print.

    '''

    top_k=neighbors_u[user][:neighbor_num] # get the top k neighbors of this user

    votes=defaultdict(int) # count the votes per movie

    for neighbor,sim_val in top_k: # for each neighbor

        for mid,pol in user_ratings[neighbor].items(): # for each movie rated by this neighbor

            if pol=='P': # positive neighbor rating
                votes[mid]+=2*sim_val
            elif pol=='N': # negative
                votes[mid]-=2*sim_val
            else: # average
                votes[mid]-=1*sim_val

    # sort the movies in desc order
    srt=sorted(votes.items(),key=lambda x:x[1], reverse=True)

    print('\nI suggest the following movies because they have\
    received positive ratings from users who tend to\nlike what you like:\n')

    cnt=0 # count number of recommendations made

    already_rated={}

    for mov, score in srt: # for each movie

        title=movies_df.loc[mov]['title'] # get the title

        rat=user_ratings[user].get(mov,None) # check if the user has already rated the movie

        if rat: # movie already rated
            already_rated[title]=rat # store the rating
            continue

        cnt+=1 # one more recommendation
        print('\n',mov, title) # print

        if cnt==rec_num:break # stop once you 've made enough recommendations

    print('\n',already_rated)


In [13]:
recommend_ub(456, movies_df, neighbors_u, user_ratings, 5, 10)



I suggest the following movies because they have    received positive ratings from users who tend to
like what you like:


 805 Time to Kill, A (1996)

 494 Executive Decision (1996)

 25 Leaving Las Vegas (1995)

 1356 Star Trek: First Contact (1996)

 376 River Wild, The (1994)

 6 Heat (1995)

 36 Dead Man Walking (1995)

 17 Sense and Sensibility (1995)

 95 Broken Arrow (1996)

 62 Mr. Holland's Opus (1995)

 {'Toy Story (1995)': 'P', 'Rock, The (1996)': 'P', 'Independence Day (a.k.a. ID4) (1996)': 'P', 'Ransom (1996)': 'P', 'Star Wars: Episode IV - A New Hope (1977)': 'P', 'Twister (1996)': 'A', 'Happy Gilmore (1996)': 'A', 'Dragonheart (1996)': 'P', 'Nutty Professor, The (1996)': 'P', 'Phenomenon (1996)': 'P', 'Grumpier Old Men (1995)': 'A', 'Bed of Roses (1996)': 'P', 'Juror, The (1996)': 'N', 'Primal Fear (1996)': 'P', 'Willy Wonka & the Chocolate Factory (1971)': 'A', 'Down Periscope (1996)': 'A', 'Jerry Maguire (1996)': 'P', 'Eraser (1996)': 'P', 'Twelve Monkeys (a.k.a. 12 

# Item-Based Recommender

In [16]:
def load_movie_ratings(ratings_df:pd.core.frame.DataFrame):

    '''
    Loads all the ratings submitted for a movie

    Returns a dictionary that maps each movie to a second dict that maps users to discretized ratings

    '''

    distinct_movies=set(ratings_df['movieId']) # get all distinct movies

    movie_ratings={} # store movie ratings per user

    for movie in distinct_movies: # for each movie

        # get the user id and rating for every rating submitted by this user
        my_ratings=ratings_df[ratings_df.movieId==movie][['userId','rating']]

        #discretize the ratings and attach them to the user
        movie_ratings[movie]=dict(zip(my_ratings.userId, my_ratings.rating.apply(discretize_rating)))

    return movie_ratings



In [17]:
movie_ratings = load_movie_ratings(ratings_df)

In [18]:
movie_ratings[88]

{6: 'N',
 45: 'P',
 89: 'N',
 91: 'A',
 226: 'A',
 274: 'P',
 276: 'P',
 307: 'P',
 414: 'N',
 492: 'P',
 539: 'P',
 555: 'P',
 577: 'A',
 585: 'P',
 599: 'N',
 608: 'N'}

In [20]:
from itertools import combinations
from collections import defaultdict

def get_movie_neighbors(movie_ratings:dict, # ratings submitted by each user
                       min_rating_num:int=5 # at least this many ratings are required for a comparison
                      ):

    '''
    Compute rating-based similarity between every two pairs of users

    '''

    #get all possible pairs of movies
    pairs=list(combinations(list(movie_ratings.keys()),2))

    msim=defaultdict(dict) # initialize the sim dictionary

    for m1,m2 in pairs: # for every movie pair

        #get a set with all the discretized ratings (movie id, polarity tuples) for m1 and m2
        s1=set([(uid,pol) for uid,pol in movie_ratings[m1].items()])
        s2=set([(uid,pol) for uid,pol in movie_ratings[m2].items()])

        # check if both movies respect the lower bound
        if len(s1)<min_rating_num or len(s2)<min_rating_num: continue

        # get the union and intersection for these two movies
        union=s1.union(s2)
        inter=s1.intersection(s2)

        # compute movies sim via the jaccard coeff
        jacc=len(inter)/len(union)

        # remember the sim values
        msim[m1][m2]=jacc
        msim[m2][m1]=jacc

    # attach each movie to its neighbors, sorted by sim in descending order
    return {movie:sorted(msim[movie].items(),key=lambda x:x[1], reverse=True) for movie in msim}




In [21]:
neighbors_m=get_movie_neighbors(movie_ratings)

In [22]:

def recommend_mb(user:int,
                 movies_df:pd.core.frame.DataFrame, # movie info
                 neighbors_u:dict, # neighbors dict
                 movie_ratings:dict, # ratings submitted per user
                 neighbor_num:int, # number of neighbors to consider
                 rec_num:int# number of movies to recommend
                ):

    my_ratings=ratings_df[ratings_df.userId==user][['movieId','rating']] # finds all the movie ratings submitted by the user

    my_liked_movies= list(my_ratings[my_ratings.rating>=4]['movieId'])

    votes=defaultdict(int) # count the votes per movie

    for mid in my_liked_movies:

      if mid not in neighbors_m:
        continue

      top_k=neighbors_m[mid][:neighbor_num] # get the top k neighbors of this movie

      for neighbor,sim_val in top_k: # for each neighbor of this movie

        votes[neighbor]+=2*sim_val

    # sort the movies in desc order
    srt=sorted(votes.items(),key=lambda x:x[1], reverse=True)

    print('\nI suggest the following movies:\n')

    cnt=0 # count number of recommendations made

    already_rated={}

    for mov, score in srt: # for each movie

        title=movies_df.loc[mov]['title'] # get the title

        rat=user_ratings[user].get(mov,None) # check if the user has already rated the movie

        if rat: # movie already rated
            already_rated[title]=rat # store the rating
            continue

        cnt+=1 # one more recommendation
        print('\n',mov, title) # print

        if cnt==rec_num:break # stop once you 've made enough recommendations

    print('\n',already_rated)


In [25]:
recommend_mb(456, movies_df, neighbors_m, movie_ratings, 5, 10)



I suggest the following movies:


 1210 Star Wars: Episode VI - Return of the Jedi (1983)

 1196 Star Wars: Episode V - The Empire Strikes Back (1980)

 494 Executive Decision (1996)

 480 Jurassic Park (1993)

 1198 Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981)

 589 Terminator 2: Judgment Day (1991)

 2571 Matrix, The (1999)

 858 Godfather, The (1972)

 6 Heat (1995)

 1994 Poltergeist (1982)

 {'Twister (1996)': 'A', 'Independence Day (a.k.a. ID4) (1996)': 'P', 'Star Wars: Episode IV - A New Hope (1977)': 'P', 'Grumpier Old Men (1995)': 'A', 'Mission: Impossible (1996)': 'P', 'Up Close and Personal (1996)': 'P', 'Dragonheart (1996)': 'P', 'Substitute, The (1996)': 'P', 'Ransom (1996)': 'P'}
